# Breast Cancer Detection Neural Network

In [135]:
import torch
import pandas as pd
from torch import nn

## Load Training Data

In [136]:
X = pd.read_csv("../data/processedData/X_train_scaled.csv", index_col=0).values
Y = pd.read_csv("../data/processedData/Y_train.csv", index_col=0).values

## Convert to Numpy Array to Tensor

In [137]:
X_tensor = torch.tensor(X, dtype=torch.float32)
Y_tensor = torch.tensor(Y, dtype=torch.float32).view(-1, 1)

In [138]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"

## Neural Network Class

In [139]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(30, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 1)
        )

    def forward(self, x):
        return self.linear_relu_stack(x)

In [140]:
model = NeuralNetwork().to(device)

logits = model(X_tensor)

In [141]:
pred_probab = torch.sigmoid(logits)

predictions = (pred_probab > 0.5).float()

correct = (predictions == Y_tensor).sum().item()

accuracy = correct / 455

In [142]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [143]:
loss_fn = nn.BCEWithLogitsLoss()

loss = loss_fn(logits, Y_tensor)

# Standard training steps
optimizer.zero_grad()
loss.backward()
optimizer.step()

In [147]:
epochs = 50000

for epoch in range(epochs):
    optimizer.zero_grad()
    logits = model(X_tensor)
    loss = loss_fn(logits, Y_tensor)
    loss.backward()
    optimizer.step()
    
    if epoch % 10 == 0:
        print(f'Epoch {epoch}, Loss: {loss.item()}')

Epoch 0, Loss: 3.6865477337499897e-09
Epoch 10, Loss: 3.6607472608807257e-09
Epoch 20, Loss: 3.6354037558083974e-09
Epoch 30, Loss: 3.6093597000075306e-09
Epoch 40, Loss: 3.5834906153553447e-09
Epoch 50, Loss: 3.5575420387345957e-09
Epoch 60, Loss: 3.5312868185144453e-09
Epoch 70, Loss: 3.5077460935895033e-09
Epoch 80, Loss: 3.4856648678527336e-09
Epoch 90, Loss: 3.4624207945199714e-09
Epoch 100, Loss: 3.4393752290640123e-09
Epoch 110, Loss: 3.4155873684937887e-09
Epoch 120, Loss: 3.3915026342867804e-09
Epoch 130, Loss: 3.3782157071726715e-09
Epoch 140, Loss: 3.365928868959145e-09
Epoch 150, Loss: 3.352389477129236e-09
Epoch 160, Loss: 3.3378961816765695e-09
Epoch 170, Loss: 3.322810693262568e-09
Epoch 180, Loss: 3.30869065479078e-09
Epoch 190, Loss: 3.2936735561150954e-09
Epoch 200, Loss: 3.2787368375863934e-09
Epoch 210, Loss: 3.262990544428135e-09
Epoch 220, Loss: 3.2551845663419954e-09
Epoch 230, Loss: 3.254060132462655e-09
Epoch 240, Loss: 3.247286883834022e-09
Epoch 250, Loss: 3.

In [145]:
X_test = pd.read_csv("../data/processedData/X_test_scaled.csv", index_col=0).values
Y_test = pd.read_csv("../data/processedData/Y_test.csv", index_col=0).values

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
Y_test_tensor = torch.tensor(Y_test, dtype=torch.float32).view(-1, 1)

In [146]:
with torch.no_grad():
    test_logits = model(X_test_tensor)
    test_pred_probab = torch.sigmoid(test_logits)
    test_predictions = (test_pred_probab > 0.5).float()
    test_correct = (test_predictions == Y_test_tensor).sum().item()
    test_accuracy = test_correct / len(Y_test_tensor)
    print(f'Test Accuracy: {test_accuracy:.4f}')

Test Accuracy: 0.9825
